In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime
import numpy as np
import time

def get_stock_data(ticker_file, start_date, end_date):
    """
    Get price data from Yahoo Finance and change to map WRDS fields
    
    Parameters:
    ticker_file (str): sp500_tickers
    start_date (str):  'YYYY-MM-DD'
    end_date (str):  'YYYY-MM-DD'
    
    Returns:
    pandas.DataFrame: price data in replace of original WRDS file
    """
    
    with open(ticker_file, 'r') as f:
        tickers = [line.strip() for line in f]
    
    all_data = []
    failed_tickers = []
    
    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            
            time.sleep(0.5)
            
            df = stock.history(start=start_date, end=end_date, interval='1d', auto_adjust=True)
            
            if df.empty:
                print(f"Warning: {ticker} No data")
                failed_tickers.append((ticker, "No data"))
                continue
            
            required_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
            if not all(col in df.columns for col in required_columns):
                print(f"Warning: {ticker} Missing required colunms")
                failed_tickers.append((ticker, "Missing required colunms"))
                continue
                
            wrds_data = pd.DataFrame({
                'tic': ticker,
                'cusip': np.nan,
                'permno': np.nan,
                'permco': np.nan,
                'issuno': 0,
                'hexcd': np.nan,
                'hsiccd': np.nan,
                'date': df.index,
                'bidlo': df['Low'],
                'askhi': df['High'],
                'prc': df['Close'],
                'adj_close_q': df['Close'],  # auto_adjust=True, Close = adj_close
                'vol': df['Volume'],
                'ret': df['Close'].pct_change(),
                'bid': df['Low'],
                'ask': df['High'],
                'shrout': np.nan,  
                'cfacpr': 1,  # auto_adjust=True, cfacpr = 1
                'cfacshr': 1,
                'openprc': df['Open'],
                'numtrd': np.nan,
                'retx': df['Close'].pct_change()
            })
            
            wrds_data['date'] = pd.to_datetime(wrds_data['date'])
            
            all_data.append(wrds_data)
            print(f"Successfully retrieved data for {ticker}")
        except Exception as e:
            print(f"Error retrieving data for {ticker}: {e}")
            failed_tickers.append((ticker, str(e)))
            continue
    
    if all_data:
        final_data = pd.concat(all_data, ignore_index=True)
        
        if failed_tickers:
            with open('failed_tickers.txt', 'w') as f:
                for ticker, reason in failed_tickers:
                    f.write(f"{ticker}: {reason}\n")
            
        return final_data
    else:
        print("No data")
        return pd.DataFrame()

def filter_abnormal_price_changes(file_path, threshold=2.0):
    """
    Filter out stocks with abnormal price changes (>threshold) between consecutive days
    
    Parameters:
    file_path (str): Path to the CSV file containing stock data
    threshold (float): Threshold for price change ratio (2.0 = 200%)
    
    Returns:
    pandas.DataFrame: Filtered data without abnormal stocks
    """
    # Read the CSV file
    print(f"Reading data from {file_path}...")
    stock_data = pd.read_csv(file_path)
    
    # Ensure date is in datetime format
    try:
        stock_data['date'] = pd.to_datetime(stock_data['date'])
    except Exception as e:
        print(f"Error converting date column: {e}")
        print("Continuing with string dates...")
    
    # Sort by ticker and date
    stock_data = stock_data.sort_values(['tic', 'date'])
    
    # Get unique tickers
    tickers = stock_data['tic'].unique()
    abnormal_tickers = set()
    
    print(f"Checking {len(tickers)} stocks for abnormal price changes...")
    
    # Check each ticker for abnormal price changes
    for ticker in tickers:
        ticker_data = stock_data[stock_data['tic'] == ticker]
        
        # Calculate absolute price change ratio
        # abs(prc_today / prc_yesterday - 1) > threshold
        ticker_data['price_change_ratio'] = ticker_data['prc'].pct_change().abs()
        
        # Check if any day has price change ratio > threshold
        if (ticker_data['price_change_ratio'] > threshold).any():
            abnormal_tickers.add(ticker)
            # Get the abnormal dates in a safer way
            abnormal_dates_df = ticker_data[ticker_data['price_change_ratio'] > threshold]
            try:
                # Try to format the dates if they're datetime objects
                if pd.api.types.is_datetime64_any_dtype(abnormal_dates_df['date']):
                    abnormal_dates = abnormal_dates_df['date'].dt.strftime('%Y-%m-%d').tolist()
                else:
                    # Otherwise just convert to string
                    abnormal_dates = abnormal_dates_df['date'].astype(str).tolist()
            except:
                # Fallback option
                abnormal_dates = ["date format issue"] * len(abnormal_dates_df)
            print(f"Abnormal price change detected for {ticker} on dates: {', '.join(abnormal_dates)}")
    
    print(f"Found {len(abnormal_tickers)} stocks with abnormal price changes (>{threshold*100}%):")
    print(", ".join(sorted(abnormal_tickers)))
    
    # Filter out abnormal tickers
    filtered_data = stock_data[~stock_data['tic'].isin(abnormal_tickers)]
    
    print(f"Original data: {len(stock_data)} rows, {len(tickers)} unique stocks")
    print(f"Filtered data: {len(filtered_data)} rows, {len(filtered_data['tic'].unique())} unique stocks")
    print(f"Removed {len(tickers) - len(filtered_data['tic'].unique())} stocks with abnormal price changes")
    
    return filtered_data

# Main execution
if __name__ == "__main__":
    # First, get the stock data from Yahoo Finance
    # stock_data = get_stock_data(
    #     './sp500_tickers.txt',
    #     start_date='1996-01-01',
    #     end_date='2025-02-10'
    # )
    # stock_data = pd.read_csv('sp500_price_199601_202502.csv')
    
    # if not stock_data.empty:
        # # Save the raw data
        # raw_file_path = 'sp500_price_199601_202502_raw.csv'
        # stock_data.to_csv(raw_file_path, index=False)
        # print(f"Successfully saved raw data, {len(stock_data['tic'].unique())} stocks in total")
        
        # Filter out stocks with abnormal price changes
        filtered_data = filter_abnormal_price_changes('sp500_price_199601_202502.csv', threshold=2.0)
        
        # Save the filtered data
        filtered_file_path = 'sp500_price_199601_202502.csv'
        filtered_data.to_csv(filtered_file_path, index=False)
        print(f"Successfully saved filtered data to {filtered_file_path}")
        
        # Save the list of abnormal tickers
        # abnormal_tickers = set(stock_data['tic'].unique()) - set(filtered_data['tic'].unique())
        # with open('abnormal_tickers.txt', 'w') as f:
        #     for ticker in sorted(abnormal_tickers):
        #         f.write(f"{ticker}\n")
        # print(f"List of abnormal tickers saved to abnormal_tickers.txt")
    # else:
    #     print("No data to process")

Reading data from sp500_price_199601_202502.csv...


C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:109: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  stock_data['date'] = pd.to_datetime(stock_data['date'])


Checking 861 stocks for abnormal price changes...


C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_data['price_change_ratio'] = ticker_data['prc'].pct_change().abs()
C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_data['price_change_ratio'] = ticker_data['prc'].pct_change().abs()
C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of

Abnormal price change detected for ACS on dates: 2007-11-27 00:00:00-05:00, 2007-12-28 00:00:00-05:00, 2008-01-02 00:00:00-05:00, 2008-01-04 00:00:00-05:00, 2008-01-14 00:00:00-05:00, 2008-01-23 00:00:00-05:00, 2008-01-25 00:00:00-05:00, 2008-01-30 00:00:00-05:00, 2008-02-20 00:00:00-05:00, 2008-02-25 00:00:00-05:00, 2008-02-27 00:00:00-05:00, 2008-03-26 00:00:00-04:00, 2008-03-31 00:00:00-04:00, 2008-04-02 00:00:00-04:00, 2008-04-09 00:00:00-04:00, 2008-04-11 00:00:00-04:00, 2008-04-15 00:00:00-04:00, 2008-04-17 00:00:00-04:00, 2008-04-23 00:00:00-04:00, 2008-04-30 00:00:00-04:00, 2008-05-05 00:00:00-04:00, 2008-07-29 00:00:00-04:00, 2008-12-23 00:00:00-05:00, 2008-12-31 00:00:00-05:00, 2009-05-01 00:00:00-04:00, 2009-05-04 00:00:00-04:00, 2009-05-14 00:00:00-04:00, 2009-05-18 00:00:00-04:00, 2009-06-02 00:00:00-04:00, 2009-06-08 00:00:00-04:00, 2009-06-29 00:00:00-04:00, 2009-07-06 00:00:00-04:00, 2009-07-24 00:00:00-04:00, 2009-08-18 00:00:00-04:00, 2009-09-03 00:00:00-04:00, 2009-0

...

Abnormal price change detected for WAMUQ on dates: 2009-04-20 00:00:00-04:00, 2010-06-02 00:00:00-04:00


C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_data['price_change_ratio'] = ticker_data['prc'].pct_change().abs()
C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_data['price_change_ratio'] = ticker_data['prc'].pct_change().abs()
C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of

Abnormal price change detected for WWY on dates: 2014-03-25 00:00:00-04:00, 2014-06-10 00:00:00-04:00, 2014-07-16 00:00:00-04:00, 2015-03-25 00:00:00-04:00, 2015-12-28 00:00:00-05:00, 2016-05-11 00:00:00-04:00, 2016-05-17 00:00:00-04:00, 2016-05-19 00:00:00-04:00, 2016-06-15 00:00:00-04:00


C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_data['price_change_ratio'] = ticker_data['prc'].pct_change().abs()
C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ticker_data['price_change_ratio'] = ticker_data['prc'].pct_change().abs()
C:\Users\misss\AppData\Local\Temp\ipykernel_34144\2218959854.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of

Found 76 stocks with abnormal price changes (>200.0%):
ACS, ASN, BAY, BDK, BFI, BLS, BLY, BMC, BOL, BRL, BSC, CBE, CFC, CFL, CGP, CIN, CNG, COMS, CPWR, CYM, CYR, DIGI, EP, EQ, FBF, FSH, GDW, GLK, GX, HET, HM, HNZ, HPC, JH, JOS, KRI, MAY, MCIC, MEE, MEL, MI, MII, MST, NCC, NGH, NLC, ORX, PALM, PBG, PCL, PLL, PRD, PTV, RAL, RBD, ROH, SBL, SGP, SLR, SOV, SRR, TIE, TIN, TNB, TOS, TRB, UCM, UPC, UPR, USBC, USH, USS, UVN, VAT, WAMUQ, WWY
Original data: 4514910 rows, 861 unique stocks
Filtered data: 4289600 rows, 785 unique stocks
Removed 76 stocks with abnormal price changes
Successfully saved filtered data to sp500_price_199601_202502_filtered.csv
